# Rz rotation via a phase gradient

Adapted from Qualtran v0.7.0, Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from IPython.display import SVG, display
from sympy import Symbol
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

## `RzViaPhaseGradient`
Apply a controlled-Rz using a phase gradient state.

Implements the following unitary action:

$$
    |\psi\rangle \otimes |x\rangle \mapsto \text{Rz}(4 \pi x) |\psi\rangle \otimes |x\rangle
$$

for every state $|\psi\rangle$ and every $x$, or equivalently

$$
    |b\rangle|x\rangle \mapsto |b\rangle e^{- (-1)^b i x/2} |x\rangle
$$

for every $b \in \{0, 1\}$ and every $x$.

To apply an $\text{Rz}(\theta) = e^{-i Z \theta/2}$, the angle register $x$ should store $\theta/(4\pi)$.

#### Parameters
 - `angle_dtype`: Data type for the `angle_data` register.
 - `phasegrad_dtype`: Data type for the phase gradient register. 

#### Registers
 - `q`: The qubit to apply Rz on.
 - `angle`: The rotation angle in radians.
 - `phase_grad`: The phase gradient register of sufficient width. 

#### References
 - [Compilation of Fault-Tolerant Quantum Heuristics for Combinatorial Optimization](https://arxiv.org/abs/2007.07391). Section II-C: Oracles for phasing by cost function. Appendix A: Addition for controlled rotations.


In [ ]:
from examples.algorithms.rz_phase_gradient_catalog import GradientRzExample

### Example Instances

In [ ]:
rz = GradientRzExample()

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Rz via phase gradient", rz.signature, {})])))

### Call Graph

In [ ]:
graph = rz.call_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
display(graph.leaf_counts)

### Append the original body

The data qubit coherently selects subtraction on zero and addition on one. The angle and negative-exponent Fourier gradient are supplied by the caller. The arrow directions of all three registers are through. Three clean carries are restored by measurement and feedback. For the angle word $a$ and gradient width $b$, the physical rotation angle is $4\pi a / 2^b$.

In [ ]:
circuit = DynamicCircuit(rz.qubit_count)
circuit.append(rz)
counts = Counter(gate.name for gate in circuit.to_ir().instructions)
print("Emitted on", rz.qubit_count, "wires", dict(counts))
print("Source logical resources", rz.resource_model())
assert counts["measure"] == 3
n = Symbol("n")
display(GradientRzExample(n, n).resource_model())